# Captura de latentes post-fine-tuning

Re-ejecuta el protocolo de la Fase 2 con el modelo fine-tuneado, usando los mismos
prompts y semillas. El VAE permanecio congelado, por lo que los latentes pre y post
son directamente comparables en el mismo espacio vectorial.

Ejecutar **una vez por modelo**. Cambiar `MODEL_KEY` y ejecutar todo.

**Prerequisito:** `ajuste_fino/entrenamiento.ipynb`

---
## 0 — Entorno

In [ ]:
from google.colab import drive
import os, sys, json
import torch

drive.mount("/content/drive", force_remount=False)
PROJECT_ROOT = "/content/drive/MyDrive/TFM/repositorio"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.environ["HF_HOME"] = "/content/.cache/huggingface"

assert torch.cuda.is_available(), "GPU no disponible. Activa T4."
print(f"GPU  : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB")

In [ ]:
import os
from huggingface_hub import login

HF_TOKEN = ""  # <-- pega tu token hf_...

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN vacio. Rellena la variable con tu token de "
        "https://huggingface.co/settings/tokens (tipo Read) "
        "y vuelve a ejecutar esta celda."
    )

login(token=HF_TOKEN, add_to_git_credential=False)
os.environ["HF_TOKEN"] = HF_TOKEN
print("Autenticado en Hugging Face.")

In [ ]:
req = os.path.join(PROJECT_ROOT, "requirements_colab.txt")
!pip install -q -U -r {req}
import diffusers, peft
print(f"diffusers={diffusers.__version__}  peft={peft.__version__}")

---
## 1 — Configuracion

> **Unico parametro a cambiar entre sesiones:** `MODEL_KEY`

In [ ]:
# ── CAMBIAR ESTO ENTRE SESIONES ──────────────────────────────────────────────
MODEL_KEY = "sd15"   # "sd15" | "sd21" | "sdxl"
# ─────────────────────────────────────────────────────────────────────────────

MODEL_CONFIGS = {
    "sd15": {"model_id": "runwayml/stable-diffusion-v1-5",
              "is_sdxl": False, "steps": 50, "guidance": 7.5, "res": 512},
    "sd21": {"model_id": "stabilityai/stable-diffusion-2-1",
              "is_sdxl": False, "steps": 50, "guidance": 7.5, "res": 768},
    "sdxl": {"model_id": "stabilityai/stable-diffusion-xl-base-1.0",
              "is_sdxl": True,  "steps": 30, "guidance": 7.5, "res": 1024},
}

cfg       = MODEL_CONFIGS[MODEL_KEY]
LORA_DIR  = os.path.join(PROJECT_ROOT, "data", "finetune", f"lora_{MODEL_KEY}")
SRC_DIR   = os.path.join(PROJECT_ROOT, "data", "latents", MODEL_KEY)
OUT_DIR   = os.path.join(PROJECT_ROOT, "data", "latents", "latents_ft", MODEL_KEY)
CKPT_PATH = os.path.join(OUT_DIR, "_ckpt_ft.pt")
os.makedirs(OUT_DIR, exist_ok=True)

assert os.path.isdir(LORA_DIR), (
    f"Adaptador LoRA no encontrado: {LORA_DIR}\n"
    f"Ejecuta primero fase5_finetune.ipynb con MODEL_KEY='{MODEL_KEY}'"
)
assert os.path.exists(os.path.join(SRC_DIR, "manifest.json")), (
    f"Manifest Fase 2 no encontrado: {SRC_DIR}\n"
    "Ejecuta primero la Fase 2 para este modelo"
)
print(f"Modelo   : {MODEL_KEY}")
print(f"LoRA     : {LORA_DIR}")
print(f"Fase 2   : {SRC_DIR}")
print(f"Salida   : {OUT_DIR}")

---
## 2 — Carga del modelo fine-tuneado

Se fusionan los pesos LoRA en el UNet base para inferencia rapida.

In [ ]:
# Carga del modelo fine-tuneado fusionando los pesos LoRA en el UNet base.
# merge_and_unload() integra los adaptadores LoRA directamente en los pesos
# del UNet (W = W₀ + BA) y elimina la estructura PEFT, produciendo un modelo
# equivalente al fine-tuneado pero sin overhead de inferencia. El resultado
# es matemáticamente idéntico a un UNet entrenado desde cero con esos pesos.
from diffusers import StableDiffusionPipeline, StableDiffusionXLPipeline
from peft import PeftModel

print(f"Cargando {cfg['model_id']} + adaptador LoRA...")
if cfg["is_sdxl"]:
    pipe = StableDiffusionXLPipeline.from_pretrained(
        cfg["model_id"], torch_dtype=torch.float16, use_safetensors=True)
else:
    pipe = StableDiffusionPipeline.from_pretrained(
        cfg["model_id"], torch_dtype=torch.float16, use_safetensors=True)

pipe.unet = PeftModel.from_pretrained(pipe.unet, LORA_DIR)
pipe.unet = pipe.unet.merge_and_unload()
pipe      = pipe.to("cuda")
pipe.set_progress_bar_config(disable=True)

print("Modelo cargado con LoRA fusionado.")
print(f"VRAM: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

---
## 3 — Captura del z0 post-fine-tuning

Mismos prompts y semillas que Fase 2. Usa `output_type='latent'` para capturar el latente directamente. Checkpoint cada 25 muestras.

In [ ]:
# Re-generación del corpus de Fase 2 con el modelo fine-tuneado.
# Se usan EXACTAMENTE los mismos prompts y semillas que en Fase 2.
# output_type='latent' hace que el pipeline devuelva z₀ directamente sin
# pasar por el VAE decoder, lo que es más eficiente que usar el hook.
# Dado que el VAE estaba congelado durante el fine-tuning, z₀_pre y z₀_ft
# son comparables en el mismo espacio vectorial — condición necesaria para
# que las métricas de deriva (Δsep, ΔPR, desplazamiento) sean interpretables.
with open(os.path.join(SRC_DIR, "manifest.json")) as f:
    src_manifest = json.load(f)

if os.path.exists(CKPT_PATH):
    ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
    latents_dict = {int(k): v for k, v in ckpt["latents_dict"].items()}
    print(f"Reanudando: {len(latents_dict)} muestras ya capturadas.")
else:
    latents_dict = {}

CKPT_EVERY = 25
device     = "cuda"
pending    = [e for e in src_manifest if e["idx"] not in latents_dict]
print(f"Pendientes: {len(pending)} / {len(src_manifest)}")

for i, entry in enumerate(pending):
    idx = entry["idx"]
    gen = torch.Generator(device=device).manual_seed(entry["seed"])

    result = pipe(
        prompt              = entry["prompt"],
        num_inference_steps = cfg["steps"],
        guidance_scale      = cfg["guidance"],
        generator           = gen,
        height              = cfg["res"],
        width               = cfg["res"],
        output_type         = "latent",
    )
    latents_dict[idx] = result.images[0].cpu()   # [4, H, W] float16

    if (i + 1) % CKPT_EVERY == 0 or (i + 1) == len(pending):
        torch.save(
            {"latents_dict": {str(k): v for k, v in latents_dict.items()}},
            CKPT_PATH,
        )
        print(f"  [{i+1:>3}/{len(pending)}] checkpoint — {len(latents_dict)} total")

print(f"\nCaptura completada: {len(latents_dict)} latentes")

---
## 4 — Guardado

In [ ]:
n = len(src_manifest)
latents_tensor = torch.stack([latents_dict[i] for i in range(n)])  # [N, 4, H, W]

torch.save(latents_tensor, os.path.join(OUT_DIR, "latents.pt"))
with open(os.path.join(OUT_DIR, "manifest.json"), "w") as f:
    json.dump(src_manifest, f, indent=2)
if os.path.exists(CKPT_PATH):
    os.remove(CKPT_PATH)

size_mb = os.path.getsize(os.path.join(OUT_DIR, "latents.pt")) / 1024**2
print(f"latents.pt    : {latents_tensor.shape}  {size_mb:.1f} MB")
print(f"manifest.json : {len(src_manifest)} entradas")
print(f"Ruta          : {OUT_DIR}")

---
## 5 — Guardado de imágenes pre/post fine-tuning

Decodifica los latentes pre (Fase 2) y post (Fase 5.3) con el VAE ya cargado
y guarda cada imagen como PNG. El VAE estaba congelado durante el fine-tuning,
por lo que es el mismo para ambas decodificaciones.

In [ ]:
import numpy as np
from PIL import Image

IMG_PRE = os.path.join(SRC_DIR,  "images")
IMG_FT  = os.path.join(OUT_DIR,  "images")
CMP_DIR = os.path.join(OUT_DIR,  "comparacion")
for d in [IMG_PRE, IMG_FT, CMP_DIR]:
    os.makedirs(d, exist_ok=True)

SCALING = pipe.vae.config.scaling_factor


def decode_and_save(vae, latents_t, manifest, out_dir, scaling, batch_size=4, label=""):
    saved = []
    n = len(latents_t)
    for start in range(0, n, batch_size):
        end   = min(start + batch_size, n)
        batch = latents_t[start:end].to("cuda", dtype=torch.float16)
        batch = batch / scaling
        with torch.no_grad():
            decoded = vae.decode(batch).sample
        decoded = decoded.cpu().float()
        decoded = (decoded / 2 + 0.5).clamp(0, 1)
        decoded = (decoded.permute(0, 2, 3, 1).numpy() * 255).astype(np.uint8)
        for j, arr in enumerate(decoded):
            idx   = manifest[start + j]["idx"]
            cat   = manifest[start + j]["category"]
            fname = f"{idx:04d}_{cat}.png"
            Image.fromarray(arr).save(os.path.join(out_dir, fname))
            saved.append(fname)
        if (end % 25 == 0) or end == n:
            print(f"  {label} [{end:>3}/{n}]")
    return saved


# Cargar latentes pre-FT (Fase 2)
Z_pre_img = torch.load(os.path.join(SRC_DIR, "latents.pt"), map_location="cpu").float()
if Z_pre_img.ndim == 5:
    Z_pre_img = Z_pre_img.squeeze(1)

print("Decodificando imágenes pre-fine-tuning...")
decode_and_save(pipe.vae, Z_pre_img, src_manifest, IMG_PRE, SCALING, label="pre")

print("\nDecodificando imágenes post-fine-tuning...")
decode_and_save(pipe.vae, latents_tensor.float(), src_manifest, IMG_FT, SCALING, label="ft")

print(f"\nImágenes pre : {IMG_PRE}")
print(f"Imágenes ft  : {IMG_FT}")

---
## 6 — Cuadrícula comparativa pre vs. post

Pares ordenados por mayor desplazamiento latente (los cambios más visibles primero).
También genera una cuadrícula independiente por cada categoría del corpus.

In [ ]:
import matplotlib.pyplot as plt
from collections import defaultdict

N_SHOW = 12   # pares en la cuadrícula global

Z_pre_flat = Z_pre_img.numpy().reshape(len(Z_pre_img), -1)
Z_ft_flat  = latents_tensor.float().numpy().reshape(len(latents_tensor), -1)
deltas_vis = np.linalg.norm(Z_ft_flat - Z_pre_flat, axis=1)
top_idx    = np.argsort(deltas_vis)[::-1][:N_SHOW]

# ── Cuadrícula global ────────────────────────────────────────────────────────
fig, axes = plt.subplots(N_SHOW, 2, figsize=(8, 4 * N_SHOW))
fig.suptitle(
    f"{MODEL_KEY.upper()} — pre / post fine-tuning\n"
    "(ordenadas por desplazamiento latente descendente)",
    fontsize=13, y=1.001,
)
for row, idx in enumerate(top_idx):
    fname = f"{src_manifest[idx]['idx']:04d}_{src_manifest[idx]['category']}.png"
    axes[row, 0].imshow(Image.open(os.path.join(IMG_PRE, fname)))
    axes[row, 1].imshow(Image.open(os.path.join(IMG_FT,  fname)))
    for ax in axes[row]: ax.axis("off")
    axes[row, 0].set_ylabel(
        f"Δ={deltas_vis[idx]:.1f}  [{src_manifest[idx]['category']}]\n"
        f"{src_manifest[idx]['prompt'][:55]}…",
        fontsize=7, rotation=0, labelpad=200, va="center",
    )
    if row == 0:
        axes[0, 0].set_title("Pre fine-tuning",  fontsize=12, fontweight="bold")
        axes[0, 1].set_title("Post fine-tuning", fontsize=12, fontweight="bold")
plt.tight_layout()
grid_path = os.path.join(CMP_DIR, f"grid_pre_vs_ft_{MODEL_KEY}.png")
plt.savefig(grid_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"Cuadrícula global guardada: {grid_path}")

# ── Cuadrículas por categoría ────────────────────────────────────────────────
MAX_PER_CAT = 8
cat_indices = defaultdict(list)
for i, entry in enumerate(src_manifest):
    cat_indices[entry["category"]].append(i)

for cat, indices in sorted(cat_indices.items()):
    sel   = indices[:MAX_PER_CAT]
    n_sel = len(sel)
    fig, axes = plt.subplots(n_sel, 2, figsize=(7, 3.5 * n_sel))
    if n_sel == 1:
        axes = axes[np.newaxis, :]
    fig.suptitle(f"{MODEL_KEY.upper()} — {cat}  (pre | post fine-tuning)", fontsize=13)
    for row, idx in enumerate(sel):
        fname = f"{src_manifest[idx]['idx']:04d}_{src_manifest[idx]['category']}.png"
        axes[row, 0].imshow(Image.open(os.path.join(IMG_PRE, fname)))
        axes[row, 1].imshow(Image.open(os.path.join(IMG_FT,  fname)))
        for ax in axes[row]: ax.axis("off")
        axes[row, 0].set_ylabel(
            f"Δ={deltas_vis[idx]:.1f}\n{src_manifest[idx]['prompt'][:50]}…",
            fontsize=7, rotation=0, labelpad=180, va="center",
        )
        if row == 0:
            axes[0, 0].set_title("Pre",  fontsize=11, fontweight="bold")
            axes[0, 1].set_title("Post", fontsize=11, fontweight="bold")
    plt.tight_layout()
    path = os.path.join(CMP_DIR, f"grid_{MODEL_KEY}_{cat}.png")
    plt.savefig(path, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"  Guardada: {path}")

## 7 — Verificación: desplazamiento latente pre→post fine-tuning

In [ ]:
# Verificación del desplazamiento latente pre→post fine-tuning.
# Calcula ‖z₀_ft - z₀_pre‖₂ para cada muestra y muestra su distribución.
# Este análisis rápido confirma que el fine-tuning ha producido cambios
# detectables antes de ejecutar el análisis cuantitativo completo (§5.4).
# Se espera: SD1.5≈37, SD2.1≈40, SDXL≈138 (SDXL sufre mayor deriva).
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

Z_pre = torch.load(os.path.join(SRC_DIR, "latents.pt"), map_location="cpu").float()
Z_ft  = latents_tensor.float()
if Z_pre.ndim == 5: Z_pre = Z_pre.squeeze(1)
if Z_ft.ndim  == 5: Z_ft  = Z_ft.squeeze(1)

Z_pre_flat = Z_pre.numpy().reshape(len(Z_pre), -1)
Z_ft_flat  = Z_ft.numpy().reshape(len(Z_ft),  -1)
deltas     = np.linalg.norm(Z_ft_flat - Z_pre_flat, axis=1)

print(f"Desplazamiento ||z_ft - z_pre||_2:")
print(f"  Media : {deltas.mean():.3f}")
print(f"  Std   : {deltas.std():.3f}")
print(f"  Min   : {deltas.min():.3f}")
print(f"  Max   : {deltas.max():.3f}")

with open(os.path.join(SRC_DIR, "manifest.json")) as f:
    mf = json.load(f)

cat_deltas = defaultdict(list)
for d, e in zip(deltas, mf):
    cat = e.get("attribute_type", "base") if e["category"] != "base" else "base"
    cat_deltas[cat].append(d)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle(f"{MODEL_KEY.upper()} — Desplazamiento latente pre->post fine-tuning")

ax = axes[0]
ax.hist(deltas, bins=30, color="#4C72B0", alpha=0.8, density=True)
ax.axvline(deltas.mean(), color="red", lw=2, ls="--", label=f"mu={deltas.mean():.1f}")
ax.set_xlabel("||z_ft - z_pre||_2"); ax.set_title("Distribucion de desplazamientos")
ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
labels = sorted(cat_deltas.keys())
ax.boxplot([cat_deltas[l] for l in labels], labels=labels,
           patch_artist=True, boxprops=dict(facecolor="#4C72B0", alpha=0.5),
           medianprops=dict(color="black", lw=2))
ax.set_ylabel("||z_ft - z_pre||_2"); ax.set_title("Desplazamiento por categoria")
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "delta_latentes.png"), dpi=120)
plt.show()